Import big matrix

In [83]:
import pandas as pd

bm_og = pd.read_csv(
    '/home/sulay/deep-linear-bandits/kuairec/data/big_matrix.csv',
        usecols=[
            'user_id',
            'video_id',
            'watch_ratio'
        ]
    ).sort_values(by=['user_id', 'video_id'])

bm_og

,user_id,video_id,watch_ratio
312,0,42,1.098951
292,0,67,2.759635
275,0,80,1.188017
61,0,110,1.408627
318,0,128,1.281867
...,...,...,...
12530627,7175,10552,0.147431
12530710,7175,10572,0.293611
12530711,7175,10572,0.293611
12530717,7175,10589,0.560359


In [75]:
bm_og.duplicated(subset=['user_id', 'video_id']).sum()

np.int64(2229837)

In [84]:
bm_f = bm_og.copy()

bm_f = bm_f.groupby(by=['user_id', 'video_id'], as_index=False).max()

bm_f

,user_id,video_id,watch_ratio
0,0,42,1.098951
1,0,67,2.759635
2,0,80,1.188017
3,0,110,1.408627
4,0,128,1.281867
...,...,...,...
10300964,7175,10519,1.107321
10300965,7175,10552,0.147431
10300966,7175,10572,0.293611
10300967,7175,10589,0.560359


In [85]:
bm_f.groupby(by='user_id').count().sort_values(by='watch_ratio')

,video_id,watch_ratio
user_id,,
3116,80,80
4604,125,125
3935,151,151
3175,155,155
268,156,156
...,...,...
6565,3341,3341
816,3348,3348
5412,3353,3353


In [86]:
bm_f[bm_f.watch_ratio < 2.0].groupby(by='user_id').count().sort_values(by='watch_ratio')

,video_id,watch_ratio
user_id,,
3116,63,63
4604,102,102
3935,121,121
3175,133,133
4812,138,138
...,...,...
6565,3295,3295
2735,3303,3303
5412,3308,3308


In [87]:
bm_f.groupby(by='video_id').count().sort_values(by='watch_ratio')

,user_id,watch_ratio
video_id,,
11,1,1
4943,1,1
40,1,1
43,1,1
10705,1,1
...,...,...
1507,5168,5168
8145,5175,5175
1037,5211,5211


Filter for the strong positive interactions to train on

In [88]:
bm = bm_f.copy()

bm = bm[bm.watch_ratio >= 2.0]

bm

,user_id,video_id,watch_ratio
1,0,67,2.759635
6,0,133,2.458447
10,0,152,2.326087
11,0,154,4.353647
15,0,171,33.276021
...,...,...,...
10300855,7175,9811,2.232981
10300862,7175,9841,2.167814
10300910,7175,10130,15.828342
10300955,7175,10408,9.241597


In [105]:
liked_counts = bm.groupby(by='user_id')['user_id'].count()

liked_counts

user_id
0       258
1       166
2        38
3       163
4        51
       ... 
7171    134
7172    271
7173     95
7174     37
7175    223
Name: user_id, Length: 7175, dtype: int64

In [106]:
liked_counts.describe()

count    7175.000000
mean      117.967108
std       101.901022
min         1.000000
25%        43.000000
50%        88.000000
75%       162.000000
max      1002.000000
Name: user_id, dtype: float64

In [89]:
bm_f[bm_f.watch_ratio < 2.0]

,user_id,video_id,watch_ratio
0,0,42,1.098951
2,0,80,1.188017
3,0,110,1.408627
4,0,128,1.281867
5,0,130,0.079565
...,...,...,...
10300964,7175,10519,1.107321
10300965,7175,10552,0.147431
10300966,7175,10572,0.293611
10300967,7175,10589,0.560359


Split into training & validation

In [231]:
from sklearn.model_selection import train_test_split

low = bm.groupby('user_id')['user_id'].transform('count') < 5

nonlow = bm[~low]
bm_train, bm_val = train_test_split(
    nonlow,
    train_size=0.8,
    shuffle=True,
    stratify=nonlow['user_id']
)
bm_train = pd.concat([bm_train, bm[low]])

bm_train, bm_val

(         user_id  video_id  watch_ratio
 966574       649      8222    15.147737
 2984327     2077       858     3.759511
 2694573     1870      9605     2.203536
 5919889     4103      9826     6.160980
 5184624     3599      4554    11.438946
 ...          ...       ...          ...
 9176402     6384      3632     3.930262
 9555229     6658      3400     2.603915
 9555261     6658      4374     2.519602
 9555297     6658      5047     2.600979
 9555519     6658     10653     2.045629
 
 [677144 rows x 3 columns],
           user_id  video_id  watch_ratio
 7733547      5350      7228     2.200787
 7830799      5415      9731     2.709779
 7411255      5131      2606     2.189596
 7130546      4935      9716     4.051970
 3694945      2579      7490     2.812920
 ...           ...       ...          ...
 9081354      6318       660     2.037965
 10267828     7151      8052     3.796250
 4945386      3435      6195     2.171728
 5397437      3744      9962     2.964363
 8280845      57

Convert to PyTorch dataset format

In [232]:
from torch.utils.data import Dataset

class KRDataset(Dataset):
    def __init__(self, data):
        self.user_ids = data['user_id'].to_numpy()
        self.item_ids = data['video_id'].to_numpy()

    def __len__(self):
        return len(self.user_ids)
    
    def __getitem__(self, idx):
        return {
            'user_id': self.user_ids[idx],
            'item_id': self.item_ids[idx]
        }

bm_train = KRDataset(bm_train)
bm_val = KRDataset(bm_val)

print(len(bm_train))
print(len(bm_val))

677144
169270


Set up two-tower

In [233]:
import torch
from torch import nn

device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"

class TwoTower(nn.Module):
    def __init__(self):
        super().__init__()

        self.u = nn.Embedding(7176, 32)
        self.i = nn.Embedding(10728, 32)

    def forward(self, u_ids, i_ids):
        u = self.u(u_ids)
        i = self.i(i_ids)

        return u @ i.T

model = TwoTower().to(device)

model

TwoTower(
  (u): Embedding(7176, 32)
  (i): Embedding(10728, 32)
)

Train the two-tower and see validation loss each epoch

In [234]:
from torch.utils.data import DataLoader

bm_train_ldr = DataLoader(bm_train, batch_size=512, shuffle=True)
bm_val_ldr = DataLoader(bm_val, batch_size=512, shuffle=True)

In [236]:
batch = next(iter(bm_train_ldr))
user_ids = batch['user_id']
item_ids = batch['item_id']

_, user_counts = torch.unique(user_ids, return_counts=True)
duplicate_users = (user_counts > 1).sum().item()

# Count duplicate items
_, item_counts = torch.unique(item_ids, return_counts=True)
duplicate_items = (item_counts > 1).sum().item()

print(f"Duplicate users: {duplicate_users}")
print(f"Duplicate items: {duplicate_items}")

Duplicate users: 28
Duplicate items: 45


In [ ]:
positive = set(zip(bm['user_id'], bm['video_id']))
total = 0
for i in range(len(user_ids)):
    false_pos = 0
    for j in range(len(item_ids)):
        if i == j:
            continue

        if (int(user_ids[i]), int(item_ids[j])) in positive:
            false_pos += 1
    
    print(f"User {i} ({user_ids[i]}) false negatives: {false_pos}")
    total += false_pos
print(f"Total false negatives: {total}")
print(f"Average per user: {total / 512}")

User 0 (4281) false negatives: 44
User 1 (1377) false negatives: 20
User 2 (5281) false negatives: 64
User 3 (5907) false negatives: 28
User 4 (3432) false negatives: 49
User 5 (1035) false negatives: 33
User 6 (4571) false negatives: 79
User 7 (309) false negatives: 18
User 8 (913) false negatives: 63
User 9 (2940) false negatives: 35
User 10 (1569) false negatives: 80
User 11 (1542) false negatives: 79
User 12 (3653) false negatives: 47
User 13 (5034) false negatives: 17
User 14 (6585) false negatives: 42
User 15 (6161) false negatives: 42
User 16 (6207) false negatives: 47
User 17 (6697) false negatives: 21
User 18 (151) false negatives: 8
User 19 (6033) false negatives: 32
User 20 (7101) false negatives: 62
User 21 (2377) false negatives: 20
User 22 (5350) false negatives: 78
User 23 (572) false negatives: 10
User 24 (3685) false negatives: 45
User 25 (6713) false negatives: 105
User 26 (1565) false negatives: 53
User 27 (1463) false negatives: 25
User 28 (4282) false negatives: 61